# D20 S1 A1 - Segmentation Metrics with NumPy
Implements binarization, pixelwise confusion counts, IoU, Dice coefficient, and soft Dice loss from scratch.


In [ ]:
import numpy as np

true = np.array([[1,1,0,0],
                 [1,1,0,0],
                 [0,0,0,0],
                 [0,0,0,0]])

pred = np.array([[1,1,1,0],
                 [1,0,0,0],
                 [0,0,0,0],
                 [0,0,0,0]])


In [ ]:
def binarize(prob, thresh=0.5):
    prob = np.asarray(prob)
    return (prob >= thresh).astype(int)


def confusion_counts(pred, true):
    pred_fg = np.asarray(pred) > 0
    true_fg = np.asarray(true) > 0

    tp = int(np.sum(pred_fg & true_fg))
    fp = int(np.sum(pred_fg & ~true_fg))
    fn = int(np.sum(~pred_fg & true_fg))
    tn = int(np.sum(~pred_fg & ~true_fg))

    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn}


def iou(pred, true):
    c = confusion_counts(pred, true)
    union = c["tp"] + c["fp"] + c["fn"]
    if union == 0:
        return 1.0
    return c["tp"] / union


def dice_coef(pred, true):
    c = confusion_counts(pred, true)
    denominator = 2 * c["tp"] + c["fp"] + c["fn"]
    if denominator == 0:
        return 1.0
    return (2 * c["tp"]) / denominator


def dice_loss(prob, true, eps=1e-6):
    prob = np.asarray(prob, dtype=float)
    true = np.asarray(true, dtype=float)

    soft = (2 * np.sum(prob * true) + eps) / (np.sum(prob) + np.sum(true) + eps)
    return 1.0 - soft


In [ ]:
c = confusion_counts(pred, true)
print(c)
print(iou(pred, true))
print(dice_coef(pred, true))
print(round(dice_loss(pred.astype(float), true.astype(float)), 4))

z = np.zeros((5, 5), dtype=int)
print(iou(z, z), dice_coef(z, z))


In [ ]:
# Extra checks matching the autograder requirements
prob = np.array([[0.49, 0.50, 0.90]])
assert np.array_equal(binarize(prob, 0.5), np.array([[0, 1, 1]]))

assert confusion_counts(pred, true) == {"tp": 3, "fp": 1, "fn": 1, "tn": 11}
assert np.isclose(iou(pred, true), 0.6)
assert np.isclose(dice_coef(pred, true), 0.75)
assert np.isclose(iou(true, true), 1.0)
assert np.isclose(dice_coef(true, true), 1.0)
assert np.isclose(dice_coef(pred, true), 2 * iou(pred, true) / (1 + iou(pred, true)))
assert np.isclose(iou(z, z), 1.0)
assert np.isclose(dice_coef(z, z), 1.0)
assert np.isclose(dice_loss(pred.astype(float), true.astype(float)), 1 - dice_coef(pred, true))
assert dice_loss(true.astype(float), true.astype(float)) < 1e-6

print("All checks passed.")
